# 03 — Transfer Learning: Taxonomy Classifier (Family/Genus/Species)\n
\n
Train a multi-head taxonomy classifier (family/genus/species) on FishNet filtered to top-100 species.\n
\n
Unknown handling: threshold on max species probability.\n
\n
Source reference for transfer learning patterns: https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

In [10]:
from pathlib import Path
import sys
import torch

ROOT = Path('..').resolve()
MANIFEST_DIR = (ROOT / 'data' / 'manifests').resolve()
OUT_DIR = (ROOT / 'outputs' / 'taxonomy_transfer').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str((ROOT / 'src').resolve()))

device = torch.device(
    'cuda'
    if torch.cuda.is_available()
    else 'mps'
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
    else 'cpu'
)
print('device:', device)
print('MANIFEST_DIR:', MANIFEST_DIR)
print('OUT_DIR:', OUT_DIR)


device: mps
MANIFEST_DIR: /Users/sushantshah/Desktop/Fish Codes/data/manifests
OUT_DIR: /Users/sushantshah/Desktop/Fish Codes/outputs/taxonomy_transfer


## Manifests expected\n
\n
This notebook expects FishNet-derived taxonomy manifests created by `01_data_build_manifests.ipynb` (FishNet section to be implemented):\n
- `data/manifests/fishnet_taxonomy_train.jsonl`\n
- `data/manifests/fishnet_taxonomy_val.jsonl`\n
- `data/manifests/fishnet_taxonomy_test.jsonl`\n
\n
Each row should include:\n
- `image_path`\n
- `taxonomy`: `{family, genus, species}`\n
\n
We will filter to **top-100 species** when generating manifests.

In [11]:
from fish_ai.data.taxonomy_dataset import FishTaxonomyDataset
from fish_ai.data.jsonl import read_jsonl
from fish_ai.models.taxonomy_classifier import TaxonomyClassifier, TaxonomyHeadSizes
from fish_ai.train.taxonomy_train import TrainConfig, build_label_maps, evaluate, train_one_epoch
from torch.utils.data import DataLoader

train_manifest = MANIFEST_DIR / 'fishnet_taxonomy_train.jsonl'
val_manifest = MANIFEST_DIR / 'fishnet_taxonomy_val.jsonl'
test_manifest = MANIFEST_DIR / 'fishnet_taxonomy_test.jsonl'

print(train_manifest.exists(), val_manifest.exists(), test_manifest.exists())


True True True


In [12]:
# Load datasets
ds_train = FishTaxonomyDataset(train_manifest, image_size=224, augment=True)
ds_val = FishTaxonomyDataset(val_manifest, image_size=224, augment=False)

# Build label maps from training rows
train_rows = []
for r in read_jsonl(train_manifest):
    tax = r['taxonomy']
    train_rows.append({'family': tax['family'], 'genus': tax['genus'], 'species': tax['species']})
maps = build_label_maps(train_rows)

sizes = TaxonomyHeadSizes(
    n_family=len(maps['family']),
    n_genus=len(maps['genus']),
    n_species=len(maps['species']),
)

sizes


TaxonomyHeadSizes(n_family=90, n_genus=198, n_species=100)

In [13]:
# Model + loaders
model = TaxonomyClassifier(sizes, backbone='efficientnet_b0', pretrained=True).to(device)
cfg = TrainConfig(num_epochs=5, batch_size=64, num_workers=2, lr=3e-4)

dl_train = DataLoader(ds_train, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
dl_val = DataLoader(ds_val, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
cfg


TrainConfig(num_epochs=5, lr=0.0003, weight_decay=0.0001, batch_size=64, num_workers=2)

In [15]:
# Train loop
history = []
for epoch in range(cfg.num_epochs):
    train_metrics = train_one_epoch(model, dl_train, opt, device=device, maps=maps)
    val_metrics = evaluate(model, dl_val, device=device, maps=maps)

    row = {'epoch': epoch + 1, **train_metrics}
    # Flatten key validation metrics for quick tables
    for lvl in ['family', 'genus', 'species']:
        for k, v in val_metrics[lvl].items():
            row[f'val_{lvl}_{k}'] = v

    history.append(row)
    print(row)

# Save checkpoint + maps
ckpt = {
    'model_state': model.state_dict(),
    'backbone': model.backbone_name,
    'maps': maps,
    'cfg': cfg.__dict__,
}
ckpt_path = OUT_DIR / 'taxonomy_efficientnet_b0.pt'
torch.save(ckpt, ckpt_path)

print('saved:', ckpt_path)

# Save run metrics for later comparison
from fish_ai.utils.run_logging import write_csv, write_json

write_csv(OUT_DIR / 'history.csv', history)
write_json(OUT_DIR / 'val_metrics_last.json', val_metrics)
print('wrote:', OUT_DIR / 'history.csv')
print('wrote:', OUT_DIR / 'val_metrics_last.json')


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/Users/sushantshah/Desktop/Fish Codes/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/sushantshah/Desktop/Fish Codes/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/Users/sushantshah/Desktop/Fish Codes/src/fish_ai/data/taxonomy_dataset.py", line 69, in __getitem__
    img = Image.open(row.image_path).convert("RGB")
          ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sushantshah/Desktop/Fish Codes/.venv/lib/python3.12/site-packages/PIL/Image.py", line 3512, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError

## Unknown threshold curve (species)\n
\n
The evaluation returns an `unknown_curve_species` object with coverage/accuracy as the threshold changes.

In [ ]:
val_metrics['unknown_curve_species']